In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.mixed_precision import set_global_policy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import datetime

# Colab için gerekli ayarlar
%matplotlib inline
plt.style.use('seaborn')

# 1. GPU Kontrolü ve Mixed Precision Ayarları
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    print("GPU bulundu:", physical_devices)
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("GPU bulunamadı, CPU kullanılacak.")

set_global_policy('mixed_float16')

# 2. CIFAR-10 Veri Setini Yükleme
(X_train, y_train), (X_test, y_test) = datasets.cifar10.load_data()

# Sınıf etiketleri
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Veri seti boyutlarını kontrol etme
print("Eğitim veri seti boyutu:", X_train.shape)
print("Test veri seti boyutu:", X_test.shape)
print("Eğitim etiket boyutu:", y_train.shape)
print("Test etiket boyutu:", y_test.shape)

# 3. Veri Ön İşleme
# Piksel değerlerini 0-1 aralığına normalize etme
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Veri artırma
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest'
)
datagen.fit(X_train)

# 4. Veri Görselleştirme
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.show()

# 5. CNN Modelini Oluşturma (Hiperparametre Optimizasyonu)
model = models.Sequential([
    # 1. Evrişim Bloğu
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # 2. Evrişim Bloğu
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # 3. Evrişim Bloğu
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.4),

    # 4. Evrişim Bloğu
    layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.4),

    # Düzleştirme ve Yoğun Katmanlar
    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax', dtype='float32')  # Mixed precision için çıktı katmanı
])

# Model özeti
model.summary()

# 6. Öğrenme Oranı Planlayıcısı
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# 7. Modeli Derleme
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 8. Modeli Eğitme
start_time = datetime.datetime.now()
history = model.fit(datagen.flow(X_train, y_train, batch_size=128),
                    epochs=100,
                    validation_data=(X_test, y_test),
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
                        lr_schedule
                    ])
end_time = datetime.datetime.now()
print("Eğitim süresi:", end_time - start_time)

# 9. Eğitim Sonuçlarını Görselleştirme
# Doğruluk grafiği
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Eğitim Doğruluğu')
plt.plot(history.history['val_accuracy'], label='Doğrulama Doğruluğu')
plt.title('Model Doğruluğu')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()

# Kayıp grafiği
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Eğitim Kaybı')
plt.plot(history.history['val_loss'], label='Doğrulama Kaybı')
plt.title('Model Kaybı')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.show()

# 10. Model Değerlendirme
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f"\nTest doğruluğu: {test_acc:.4f}")
print(f"Test kaybı: {test_loss:.4f}")

# 11. Tahminler ve Hata Matrisi
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = y_test.flatten()

# Hata matrisi
cm = confusion_matrix(y_test_classes, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Hata Matrisi')
plt.ylabel('Gerçek Etiket')
plt.xlabel('Tahmin Edilen Etiket')
plt.show()

# Sınıflandırma raporu
print("\nSınıflandırma Raporu:")
print(classification_report(y_test_classes, y_pred_classes, target_names=class_names))

# 12. Örnek Tahmin Görselleştirme
plt.figure(figsize=(15, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_test[i])
    plt.title(f"Tahmin: {class_names[y_pred_classes[i]]}\nGerçek: {class_names[y_test_classes[i]]}")
    plt.axis('off')
plt.show()

# 13. Modeli Kaydetme
model.save('cifar10_cnn_hypertuned_model.h5')
print("Model kaydedildi: cifar10_cnn_hypertuned_model.h5")